# Structured text and line media - JavaScript

All 8 JavaScript examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## Text media and Arrow batches

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
fs.writeFileSync(path.join(root, 'app.log'), 'first event\nsecond event\n')

const table = new IOBase(path.join(root, 'app.log')).readArrowReader().intoTable()
assert.equal(table.numRows, 2)
assert.deepEqual([...table.getChild('message')], ['first event', 'second event'])

const target = new IOBase(path.join(root, 'copy.log'))
target.overwriteArrowTable(table)
target.appendArrowTable(table)
assert.equal(
  fs.readFileSync(path.join(root, 'copy.log'), 'utf8'),
  'first event\nsecond event\nfirst event\nsecond event\n',
)

assert.throws(
  () => target.mergeArrowTable(table, target.recordOptions().withMergeByNames(['message'])),
  /row identity/,
)

fs.rmSync(root, { recursive: true, force: true })

### Line iteration with `Text`

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log')
fs.writeFileSync(target, 'first event\nsecond event\n')

const handle = new IOBase(target)
assert.deepEqual([...handle.readLines()], ['first event', 'second event'])

fs.rmSync(root, { recursive: true, force: true })

## Raw shared-Value access

In [ ]:
const assert = require('node:assert/strict')
const { Value, json } = require('yggdryl')

const quote = json.loads('{"symbol":"AAPL","price":12.5}', { value: true })

assert.ok(quote instanceof Value)
assert.equal(quote.get('symbol').asUtf8(), 'AAPL')
assert.equal(quote.path('price').kind, 'f64')
assert.equal(quote.set('venue', 'XNAS').get('venue').asUtf8(), 'XNAS')
assert.deepEqual(quote.asJs(), { price: 12.5, symbol: 'AAPL' })

### Typed `Value` families

In [ ]:
const assert = require('node:assert/strict')
const { Value } = require('yggdryl')

assert.equal(Value.fromJs(40).add(2).asJs(), 42)
assert.ok(Value.d128(1n, 0).divide(Value.d128(2n, 0)).equals(Value.d128(5n, 1)))

## Field-directed parsing

In [ ]:
const assert = require('node:assert/strict')
const { Field, json } = require('yggdryl')

const amount = new Field('amount', 'decimal128(8, 2)', false)
const value = json.loads('"12.50"', { field: amount, value: true })

assert.equal(value.kind, 'd128')
assert.equal(value.unscaled, 1250n)
assert.equal(value.scale, 2)

## Raw document codecs

In [ ]:
const assert = require('node:assert/strict')
const { Value, codec } = require('yggdryl')

const value = codec.from('{"id":1}', { value: true })

assert.ok(value instanceof Value)
assert.equal(value.get('id').kind, 'u64')
assert.deepEqual(codec.into(value, { format: 'json' }), Buffer.from('{"id":1}'))

## Formatting

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const pretty = json.dumps({ id: 1 }, { indent: 2 })
const compact = json.dumps({ id: 1 }, { indent: null })

assert.deepEqual(pretty, Buffer.from('{\n  "id": 1\n}'))
assert.deepEqual(compact, Buffer.from('{"id":1}'))

## Placeholders

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('yggdryl')

const document = 'host: "{{ HOST }}"\nport: "{{ PORT | default(8080) }}"\n'
const value = yaml.loads(document, {
  placeholders: { HOST: 'db.internal' },
})

assert.deepEqual(value, { host: 'db.internal', port: 8080 })